# 上下文管理器

学习目标：理解 with 的进入、退出与异常处理过程，编写上下文管理器，并用 ExitStack 管理数量不固定的资源。

前置知识：类与特殊方法、异常处理、文件读写、生成器和装饰器。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

文件示例在临时目录中运行并自动清理。

## 1 从资源关闭到 with

### 1.1 让退出操作跟随代码块

文件读写已经使用过 with。本章进一步解释它如何调用进入、退出方法，以及如何处理异常。

下面用 StringIO 表示内存中的文本流：在 with 中读取，离开后关闭。with 不创建新的作用域，名称仍然存在，但已关闭的流不能继续读写。

In [1]:
from io import StringIO

with StringIO("Python\n") as stream:
    print(stream.readline().strip())  # Python
    print(stream.closed)  # False：代码块中仍可使用。

print(stream.closed)  # True：名称仍在，资源已经关闭。

Python
False
True


### 1.2 异常退出时也要清理

对于文件和 StringIO，代码块抛出异常时，with 仍会关闭流，然后让异常向外传播。需要处理业务错误时，在外层使用相应的 except。

这相当于把重复的清理职责交给资源对象；并不表示所有上下文管理器都会抑制异常。

In [2]:
stream = StringIO("80")

try:
    with stream:
        score = int(stream.read())
        raise ValueError("演示读取后的业务错误")
except ValueError as error:
    print(str(error))  # 演示读取后的业务错误

print(stream.closed)  # True：业务失败没有跳过关闭操作。

演示读取后的业务错误
True


## 2 上下文管理协议

### 2.1 进入方法与退出方法

上下文管理器（context manager）通过两个特殊方法配合 with；它是一种对象协议，不要求继承某个特定类。

| 方法 | 中文含义／职责 |
| --- | --- |
| \_\_enter\_\_(self) | 进入上下文，返回 as 后名称接收的值 |
| \_\_exit\_\_(self, exc_type, exc_value, traceback) | 退出上下文，执行清理并决定是否抑制异常 |

exc_type、exc_value、traceback 分别表示异常类型、异常对象和回溯；没有异常时，三个值都是 None。as 接收的是进入方法的返回值，不一定是管理器本身。

下面的管理器负责创建和关闭文本流；输出用于观察调用顺序。

In [3]:
class TextSession:
    """为一个 with 代码块创建并关闭文本流。"""

    def __init__(self, text):
        self.text = text
        self.stream = None

    def __enter__(self):
        self.stream = StringIO(self.text)
        print("进入", self.text)  # 本例先显示“进入 A”；后续示例显示各自传入的文本。
        return self.stream

    def __exit__(self, exc_type, exc_value, traceback):
        self.stream.close()
        status = "正常" if exc_type is None else exc_type.__name__
        print("退出", self.text, status)  # 本例显示“退出 A 正常”；异常退出时末项为异常类名。
        return False


session = TextSession("A")
with session as stream:
    print(stream is session.stream, stream is session)  # True False。
    print(stream.read())  # A。

print(stream.closed)  # True；此前已完成退出提示。
# 进入 A
# True False：as 得到文本流，而不是 session。
# A
# 退出 A 正常
# True

进入 A
True False
A
退出 A 正常
True


### 2.2 退出方法的返回值

有异常时，退出方法返回真值表示抑制异常，返回假值（包括 None）表示继续传播。没有异常时，其返回值被忽略。

退出方法不需要重新抛出传入的异常；完成清理并返回假值即可。下面只允许忽略 ValueError 及其子类，其他异常仍向外传播。

In [4]:
class IgnoreValueError(TextSession):
    """关闭文本流，并仅抑制 ValueError 及其子类。"""

    def __exit__(self, exc_type, exc_value, traceback):
        super().__exit__(exc_type, exc_value, traceback)
        return exc_type is not None and issubclass(exc_type, ValueError)


with IgnoreValueError("B"):
    raise ValueError("本例允许忽略")

print("继续执行")  # 前面的 ValueError 已被抑制。

try:
    with IgnoreValueError("C"):
        raise KeyError("missing")
except KeyError as error:
    print(type(error).__name__)  # KeyError：没有被误当成成功。
# 两次都先输出“进入”，再输出“退出”和异常类型。

进入 B
退出 B ValueError
继续执行
进入 C
退出 C KeyError
KeyError


### 2.3 进入失败时由谁清理

只有进入方法成功返回，with 才会负责调用退出方法；给 as 目标赋值时发生错误，也属于需要退出的情况。

如果进入方法在取得资源后、返回前失败，需要自行清理已经取得的资源。下面的参数校验故意安排在取得流之后，用于观察这条边界。

In [5]:
class CheckedTextSession(TextSession):
    """进入时检查文本，失败则立即关闭刚创建的流。"""

    def __enter__(self):
        stream = super().__enter__()
        try:
            if not self.text:
                raise ValueError("文本不能为空")
        except ValueError:
            stream.close()
            raise
        return stream


session = CheckedTextSession("")
try:
    with session:
        print("不会执行")  # __enter__ 先报错，本行没有输出。
except ValueError as error:
    print(str(error))  # 文本不能为空。

print(session.stream.closed)  # True：由 __enter__ 自行关闭。
# 输出中没有“退出”：__enter__ 抛出异常，with 未调用 __exit__。

进入 
文本不能为空
True


## 3 多个管理器与退出顺序

### 3.1 从左到右进入，从右到左退出

一个 with 可以包含多个管理器，效果相当于嵌套的 with。先进入的资源最后退出，后面的管理器也可以使用前面 as 绑定的值。

每次新建 TextSession，避免嵌套使用同一个实例时覆盖其尚未关闭的 stream。

In [6]:
with TextSession("外层") as outer, TextSession("内层") as inner:
    print(outer.read(), inner.read())

# 进入 外层
# 进入 内层
# 外层 内层
# 退出 内层 正常
# 退出 外层 正常

进入 外层
进入 内层
外层 内层
退出 内层 正常
退出 外层 正常


### 3.2 后面的进入失败，前面的仍需退出

后一个管理器进入失败时，已经成功进入的管理器会按嵌套规则退出。失败的管理器自身仍遵循上一节的清理责任。

In [7]:
try:
    with TextSession("已取得"), CheckedTextSession(""):
        print("不会执行")
except ValueError:
    print("进入失败已传出")
# 进入 已取得
# 进入
# 退出 已取得 ValueError
# 进入失败已传出

进入 已取得
进入 
退出 已取得 ValueError
进入失败已传出


### 3.3 函数返回前的退出

return、break 等离开代码块的情况也会触发退出；它们不是异常，因此退出方法收到的三个异常参数都是 None。

下面先计算返回值，再完成退出操作，最后把返回值交给调用者。

In [8]:
def read_first_character():
    """返回第一个字符，离开函数前由 with 关闭流。"""
    with TextSession("Python") as stream:
        return stream.read(1)


print(read_first_character())  # 先输出“退出 Python 正常”，再输出 P。

进入 Python
退出 Python 正常
P


## 4 用生成器编写上下文管理器

### 4.1 contextmanager 与一次 yield

contextlib.contextmanager 是装饰器：它把生成器函数转换为创建上下文管理器的工厂函数。普通生成器本身不因此自动支持 with。

生成器必须恰好 yield 一次：yield 前取得资源，yield 的值交给 as，代码块退出后继续运行生成器。清理操作放在 finally 中，正常结束和异常结束都能执行。

In [9]:
from contextlib import contextmanager


@contextmanager
def text_stream(text):
    """提供临时文本流，离开上下文时关闭。"""
    stream = StringIO(text)
    try:
        # yield 暂停期间资源交给 with 使用，恢复后进入关闭流程。
        yield stream
    finally:
        stream.close()
        print("流已关闭")


with text_stream("80") as stream:
    print(stream.read())  # 80

print(stream.closed)  # True，前面已输出“流已关闭”。

try:
    with text_stream("90") as stream:
        raise ValueError("演示处理失败")
except ValueError:
    print("错误已传出")

print(stream.closed)  # True：异常路径也先关闭，再传播。

80
流已关闭
True
流已关闭
错误已传出
True


### 4.2 异常在 yield 处重新出现

with 代码块中的未处理异常，会在生成器暂停的 yield 位置重新抛出。因此可以在 yield 周围使用 except 和 finally。

如果捕获异常只是为了记录或补充处理，完成后必须重新抛出；捕获后正常结束，会被视为已经处理该异常。

In [10]:
@contextmanager
def report_value_error():
    """记录数值错误后继续传播，并观察结束顺序。"""
    try:
        yield
    except ValueError as error:
        print("记录：", str(error))
        raise
    finally:
        print("上下文结束")


try:
    with report_value_error():
        int("bad")
except ValueError:
    print("外层仍收到 ValueError")
# 顺序：记录错误 → 上下文结束 → 外层收到错误。
# 这里保留裸 raise，避免把处理失败伪装成成功。

记录： invalid literal for int() with base 10: 'bad'
上下文结束
外层仍收到 ValueError


### 4.3 不能用多次 yield 表示多个代码块

一次 with 只对应一次进入和一次退出。生成器没有产出值，或者在退出时再次产出值，都会违反 contextmanager 的约定。

下面是用于观察错误的反例，不是批量处理数据的写法。需要多次产出数据时，应使用普通生成器和 for。

In [11]:
@contextmanager
def too_many_values():
    """反例：第二次 yield 违反上下文管理器约定。"""
    yield "first"
    yield "second"


try:
    with too_many_values() as value:
        print(value)  # first：代码块已经执行。
except RuntimeError as error:
    print(type(error).__name__)  # RuntimeError：退出时发现第二次 yield。

first
RuntimeError


### 4.4 工厂函数可重复调用，单个实例只用一次

text_stream 是可重复调用的工厂；每次调用返回一个新的管理器。不要把其中一个返回对象保存下来，再反复交给 with。

contextmanager 创建的管理器还可作为装饰器：每次调用被装饰函数时会新建生成器实例。此时没有 as 绑定，函数不会自动收到 yield 的值。

In [12]:
for text in ("A", "B"):
    with text_stream(text) as stream:
        print(stream.read())
# 分别输出 A、B，每次退出都会输出“流已关闭”。


@report_value_error()
def parse_score(text):
    """把成绩文本转换为整数。"""
    return int(text)


print(parse_score("80"))
print(parse_score("90"))
# 两次分别先输出“上下文结束”，再输出 80、90。
# 每次调用拥有独立的生成器实例。

A
流已关闭
B
流已关闭
上下文结束
80
上下文结束
90


## 5 用 ExitStack 管理多个资源

### 5.1 资源数量由输入决定

固定数量的资源可以直接写多个 with 项；文件数量由输入决定时，contextlib.ExitStack 可逐个进入并登记退出操作。

| 方法 | 中文含义／职责 |
| --- | --- |
| enter_context(cm) | 进入管理器 cm，登记其退出方法，返回进入方法的结果 |
| callback(func, \*args, \*\*kwargs) | 登记普通清理函数及实参 |
| close() | 立即执行已登记的退出操作，按无异常状态退出 |
| pop_all() | 把已登记的退出操作转移给一个新的 ExitStack |

cm 表示上下文管理器；func 表示清理函数，args 和 kwargs 是传给它的位置实参与关键字实参。退出操作按登记的逆序执行，应使用 with 或显式 close，不依赖垃圾回收触发。

In [13]:
from contextlib import ExitStack

streams = []
with ExitStack() as stack:
    for text in ("80", "90", "100"):
        stream = stack.enter_context(StringIO(text))
        streams.append(stream)
    print(sum(int(stream.read()) for stream in streams))  # 270

print([stream.closed for stream in streams])  # [True, True, True]

270
[True, True, True]


### 5.2 后续文件打不开时，关闭前面已打开的文件

每个资源取得后立即登记，后续步骤失败时，已登记的退出操作仍会执行。

下面先创建一个小文件，再尝试打开一个不存在的文件。外层临时目录负责清理测试数据，内层 ExitStack 负责先关闭文件。

In [14]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    folder = Path(directory)
    present = folder / "present.txt"
    present.write_text("80\n", encoding="utf-8")

    opened = []
    try:
        with ExitStack() as stack:
            for path in (present, folder / "missing.txt"):
                stream = stack.enter_context(path.open(encoding="utf-8"))
                opened.append(stream)
    except FileNotFoundError:
        print("第二个文件不存在")

    print(len(opened), opened[0].closed)  # 1 True

print(folder.exists())  # False：文件已关闭，临时目录也已清理。

第二个文件不存在
1 True
False


### 5.3 登记普通清理函数

callback 用于已有清理函数的资源，不要求资源实现上下文管理协议。传入函数对象和实参，不要提前调用函数。

callback 登记的函数不会收到异常信息，其返回值不能抑制异常。若需要参与异常处理，应使用实现了退出协议的管理器。

In [15]:
events = []


def record_cleanup(events, label):
    """记录清理顺序；返回值用于观察 callback 不抑制异常。"""
    events.append(label)
    return True


try:
    with ExitStack() as stack:
        stack.callback(record_cleanup, events, "先登记")
        stack.callback(record_cleanup, events, "后登记")
        raise ValueError("业务失败")
except ValueError:
    print("ValueError 继续传播")

print(events)  # ['后登记', '先登记']

ValueError 继续传播
['后登记', '先登记']


### 5.4 转移清理责任

pop_all 不立即执行清理，而是返回持有这些退出操作的新 ExitStack。原来的栈退出后，资源仍可使用；接手的新栈必须负责关闭。

这适合“全部取得成功后，把资源交给下一段代码”的场景。下面用外层 with 管理接手者，保证转移后仍有明确的退出位置。

In [16]:
with ExitStack() as owner:
    with ExitStack() as pending:
        stream = pending.enter_context(StringIO("已移交"))
        transferred = pending.pop_all()
        owner.enter_context(transferred)

    print(stream.closed)  # False：pending 已不再持有关闭责任。
    print(stream.read())  # 已移交

print(stream.closed)  # True：owner 退出时关闭接手的新栈。

False
已移交
True


## 6 常用上下文工具

### 6.1 closing：为 close 方法安排退出

contextlib.closing 会在退出时调用对象的 close，适合有关闭方法、但不支持 with 的对象。已经支持 with 的文件对象通常直接使用即可。

生成器有 close 方法，但普通生成器不是上下文管理器；closing 可以让提前结束迭代时的关闭责任落在 with 上。

In [17]:
from contextlib import closing


def iter_lines(text):
    """逐行产出文本，并在生成器结束或关闭时释放流。"""
    stream = StringIO(text)
    try:
        for line in stream:
            yield line.strip()
    finally:
        stream.close()
        print("生成器中的流已关闭")


with closing(iter_lines("A\nB\n")) as lines:
    for line in lines:
        print(line)  # A
        break

# 随后输出“生成器中的流已关闭”。
# 关闭由 with 退出触发，不依赖 break 自动关闭生成器。

A
生成器中的流已关闭


### 6.2 suppress：只忽略明确允许的异常

contextlib.suppress 按指定异常类型及其子类抑制异常，然后从整个 with 之后继续；不会回到代码块中出错语句的下一行。

只把它用于“忽略该错误仍然正确”的小范围操作。Python 3.12 也支持从异常组中移除匹配的异常，未匹配的部分仍会传播。

In [18]:
from contextlib import suppress

settings = {"mode": "fast"}

with suppress(KeyError):
    del settings["cache"]
    print("不会执行")  # 删除缺失键已抛出 KeyError，本行不执行。

print(settings)  # {'mode': 'fast'}：本例允许待删除项本来就不存在。
# 单独删除字典键时，也可用 settings.pop("cache", None) 表达这一需求。
# 不要用 suppress(Exception) 包住整个业务流程。

{'mode': 'fast'}


### 6.3 nullcontext：使用资源，但不接管关闭

contextlib.nullcontext 返回一个不做额外进入、退出操作的管理器，并把传入对象交给 as。

这可以明确区分资源归属：函数自己打开的文件由函数关闭，调用者传入的流仍由调用者关闭。下面只演示后一种情况，不再重复文件读取。

In [19]:
from contextlib import nullcontext


def first_line(stream):
    """读取调用者提供的流，不接管其关闭责任。"""
    with nullcontext(stream) as current:
        return current.readline().strip()


with StringIO("A\nB\n") as supplied:
    print(first_line(supplied))  # A
    print(supplied.closed)  # False：first_line 未关闭调用者的流。
    print(supplied.readline().strip())  # B

print(supplied.closed)  # True：调用者自己的 with 最终关闭。

A
False
B
True


## 7 复用与嵌套的边界

“可复用”表示同一个管理器可先后用于多次 with；“可重入”表示它还能在尚未退出时被嵌套使用。可重入的管理器也是可复用的，两者不能等同。

ExitStack 可先后复用，但不适合嵌套使用同一个实例：内层退出就会执行该实例中全部已登记的清理。需要嵌套时分别创建实例。

In [20]:
stack = ExitStack()
events = []

# 先故意嵌套同一个栈，再用两个独立栈对照清理边界。
with stack:
    stack.callback(events.append, "外层")
    with stack:
        stack.callback(events.append, "内层")
    print(events)  # ['内层', '外层']：内层退出已经清空同一个栈。

events = []
with ExitStack() as outer:
    outer.callback(events.append, "外层")
    with ExitStack() as inner:
        inner.callback(events.append, "内层")
    print(events)  # ['内层']：外层登记的操作尚未执行。

print(events)  # ['内层', '外层']

['内层', '外层']
['内层']
['内层', '外层']


## 本章小结

（1）with 调用进入、退出方法；as 接收进入方法的返回值。进入失败时，要自行清理已经取得的资源。

（2）退出方法负责清理；仅在明确允许时返回真值抑制异常。多个管理器按进入的逆序退出。

（3）contextmanager 使用恰好一次 yield 连接代码块，finally 负责清理；只记录异常时要重新抛出。

（4）ExitStack 逐个登记退出操作，callback 登记普通清理函数，pop_all 转移清理责任。

（5）资源归属应清楚：谁创建或接管，谁负责关闭；同一管理器能否复用、嵌套，需要看其约定。

## 练习

（1）先预测下面 events 的完整内容，再执行核对。解释两个退出操作为什么按这个顺序出现。

In [21]:
events = []

with ExitStack() as stack:
    stack.callback(events.append, "先登记")
    stack.callback(events.append, "后登记")
    events.append("处理")

print(events)
# 核对时分别标出业务操作和退出操作。

['处理', '后登记', '先登记']


（2）实现 managed_text(text)，返回可用于 with 的管理器；进入时提供 StringIO，正常退出或代码块抛出 ValueError 时都关闭流，且不抑制 ValueError。

使用 contextmanager、yield 和 finally；分别检查正常读取及异常退出后的 closed 状态。

In [22]:
# 在这里定义 managed_text，并分别运行正常与异常两种情况。
# 正常：读取 "Python" 得到 "Python"，退出后 closed 为 True。
# 异常：外层 except ValueError 能收到错误，且流已关闭。

（3）用 ExitStack 同时管理 texts 中的文本流，读取每个流中的整数，返回列表。检查返回值为 [10, 20, 30]，所有流在 with 退出后都已关闭。

再用 ["10", "bad", "30"] 运行；ValueError 应向外传播，同时已经登记的流全部关闭。

In [23]:
texts = ["10", "20", "30"]

# 在这里创建 ExitStack、登记流并转换内容。
# 保留流对象的列表，用 closed 检查正常和异常路径。
# 创建与登记紧邻进行，避免有资源取得后却未被纳入关闭范围。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [语言参考：with 的完整执行过程与多项嵌套](https://docs.python.org/zh-cn/3.12/reference/compound_stmts.html#the-with-statement)；[代码块与作用域](https://docs.python.org/zh-cn/3.12/reference/executionmodel.html#structure-of-a-program)、[生成器关闭](https://docs.python.org/zh-cn/3.12/reference/expressions.html#generator.close)；[进入方法](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__enter__)、[退出方法及异常参数](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__exit__)；[contextmanager](https://docs.python.org/zh-cn/3.12/library/contextlib.html#contextlib.contextmanager)、[ExitStack](https://docs.python.org/zh-cn/3.12/library/contextlib.html#contextlib.ExitStack)、[enter_context](https://docs.python.org/zh-cn/3.12/library/contextlib.html#contextlib.ExitStack.enter_context)、[callback](https://docs.python.org/zh-cn/3.12/library/contextlib.html#contextlib.ExitStack.callback)、[close](https://docs.python.org/zh-cn/3.12/library/contextlib.html#contextlib.ExitStack.close)、[pop_all](https://docs.python.org/zh-cn/3.12/library/contextlib.html#contextlib.ExitStack.pop_all)、[closing](https://docs.python.org/zh-cn/3.12/library/contextlib.html#contextlib.closing)、[suppress](https://docs.python.org/zh-cn/3.12/library/contextlib.html#contextlib.suppress)、[nullcontext](https://docs.python.org/zh-cn/3.12/library/contextlib.html#contextlib.nullcontext)、[一次性、可复用与可重入管理器](https://docs.python.org/zh-cn/3.12/library/contextlib.html#single-use-reusable-and-reentrant-context-managers)；[StringIO](https://docs.python.org/zh-cn/3.12/library/io.html#io.StringIO)、[流关闭](https://docs.python.org/zh-cn/3.12/library/io.html#io.IOBase.close)；[TemporaryDirectory](https://docs.python.org/zh-cn/3.12/library/tempfile.html#tempfile.TemporaryDirectory)、[Path.write_text](https://docs.python.org/zh-cn/3.12/library/pathlib.html#pathlib.Path.write_text)、[Path.open](https://docs.python.org/zh-cn/3.12/library/pathlib.html#pathlib.Path.open)。 |